# Studying the final neutron star population

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad

# Set `usetex=False' if you do not have LaTeX installed.
rc('text', usetex=False)
rc('font', family='serif')
mpl.rcParams['text.latex.preamble'] = [r"\usepackage{amsmath}"]

In [ ]:
rcParams["mathtext.fontset"] = "stix"
# rcParams["font.family"] = "Liberation serif"
rcParams["font.size"] = "22"
# rcParams['font.weight']='bold'
rcParams["figure.figsize"] = "8.0, 8.0"
rcParams["figure.autolayout"] = "False"

rcParams["axes.linewidth"] = "1.7"
rcParams["axes.labelpad"] = "15.0"
rcParams["axes.titlepad"] = "15.0"

rcParams["xtick.direction"] = "in"
rcParams["xtick.top"] = True
rcParams["xtick.major.pad"] = "10.0"
rcParams["xtick.minor.pad"] = "10.0"
rcParams["xtick.major.size"] = "10.0"
rcParams["xtick.major.width"] = "1.7"
rcParams["xtick.minor.size"] = "5.0"
rcParams["xtick.minor.width"] = "1.7"
rcParams["xtick.labelsize"] = "25"

rcParams["ytick.direction"] = "in"
rcParams["ytick.right"] = True
rcParams["ytick.major.pad"] = "10.0"
rcParams["ytick.minor.pad"] = "10.0"
rcParams["ytick.major.size"] = "10.0"
rcParams["ytick.major.width"] = "1.7"
rcParams["ytick.minor.size"] = "5.0"
rcParams["ytick.minor.width"] = "1.7"
rcParams["ytick.labelsize"] = "25"

Select an `final_population.pkl.gz` file to import:

In [ ]:
data = pd.read_pickle("../data/simulation_maxwell_sigma265_h018/final_population.pkl.gz", compression="gzip")
data.head()

In [ ]:
x = data["x"]["[kpc]"].to_numpy()
y = data["y"]["[kpc]"].to_numpy()
z = data["z"]["[kpc]"].to_numpy()
RA = data["RA"]["[deg]"].to_numpy()
DEC = data["DEC"]["[deg]"].to_numpy()
v_RA = data["v_RA"]["[mas/yr]"].to_numpy()
v_DEC = data["v_DEC"]["[mas/yr]"].to_numpy()
v_r = data["v_r"]["[km/s]"].to_numpy()
v_phi = data["v_phi"]["[km/s]"].to_numpy()
v_z = data["v_z"]["[km/s]"].to_numpy()

Top view of the galactic plane

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    y,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=1,
    alpha=0.2,
    rasterized=False
)

ax.plot(0.0, 8.3, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.show()

Side view of the galactic plane

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    z,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=1,
    alpha=0.3,
    rasterized=True,
)

ax.plot(0.0, 0.02, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.show()

Histrogramming the pulsars radial position and comparing to underlying initial position PDF

In [ ]:
def pdf_r(r: float) -> float:
    """
    The Milky Way's stellar radial density in the galactic plane according
    to eq. (15) of Yusifov & Küçük (2004).

    Args:
        r (float): distance from the galactic center in [kpc].

    Returns:
        float: stellar radial density in [1/kpc].
    """

    # Here we keep R_sun = 8.5 kpc for consistency with the results
    # of Yusifov & Küçük (2004)
    rsun = 8.5  # Sun's distance from the galactic center in [kpc].
    A = 37.6  # +- 1.90 [1/kpc^2]
    a = 1.64  # +-0.11
    b = 4.01  # +-0.24
    r1 = 0.55  # +- 0.10 [kpc]

    # Stellar surface density following eq. (15) of Yusifov & Küçük (2004).
    rho = (
        A
        * ((r + r1) / (rsun + r1)) ** a
        * np.exp(-b * (r - rsun) / (rsun + r1))
    )

    # Multiply the stellar surface density with the area element in polar coordinates.
    pdf_r = 2 * np.pi * r * rho

    return pdf_r

For normalization purposes, determine the area underneath the theoretical PDF curve:

In [ ]:
pdf_area = quad(pdf_r, 0, 100)[0]
print(pdf_area)

In [ ]:
r = np.sqrt(x**2 + y**2)
r_bins = np.linspace(0.0, 30.0, 51)

In [ ]:
fig, ax = plt.subplots()

ax.hist(
    r,
    bins=r_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=0.7,
    label="Simulation evolved",
    density=True
)
ax.plot(
    r_bins,
    pdf_r(r_bins) / pdf_area,
    linestyle="-",
    lw=4,
    color="tab:red",
    alpha=0.7,
    label="initial YK04",
)
plt.xlabel(r"$r$ [kpc]")
plt.ylabel(r"Normalized radial PDF")
plt.xlim(0.0, 30.0)
plt.legend(frameon=False, loc=0)

plt.show()

Histrogramming the pulsars $z$ position and comparing to underlying PDF

In [ ]:
def pdf_z(z: float) -> float:
    """
    Probability density function for the height from the galactic equatorial plane
    according to eq. (2) in Gullon et al. (2014).

    Args:
        z (float): distance from the galactic plane in [kpc].

    Returns:
        float: distribution of stars per kpc in z direction.
    """

    # We use an exponential distribution as given by Wainscoat et al. (1992)
    # and choose a mean scale height characteristic for a young distribution as
    # obtained by Gullon et al. (2014).

    h_c = 0.18
    pdf_z = 1.0 / h_c * np.exp(-z / h_c)

    return pdf_z

In [ ]:
z_bins = np.linspace(0.0, 5.0, 51)

In [ ]:
fig, ax = plt.subplots()

ax.hist(
    z,
    bins=z_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=0.7,
    label="Simulation evolved",
    density=True
)
ax.plot(
    z_bins,
    pdf_z(z_bins),
    linestyle="-",
    lw=4,
    color="tab:red",
    alpha=0.7,
    label="Theoretical exp",
)
plt.xlabel(r"$z$ [kpc]")
plt.ylabel(r"Normalized height PDF")
plt.xlim(0.0, 5.0)
plt.legend(frameon=False, loc=1)

plt.show()

In [ ]:
fig, ax = plt.subplots()
x_bins = np.linspace(-2000.,2000.,50)  

ax.hist(
    v_r,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=0.5,
    label=r"$v_r$",
)
ax.hist(
    v_phi,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=0.5,
    label=r"$v_{\phi}$",
)
ax.hist(
    v_z,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=0.5,
    label=r"$v_z$",
)
ax.set_xlabel(r"Velocity components [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.legend(frameon=False, loc=0)

plt.show()

Distribution of the total velocity magnitude

In [ ]:
def pdf_kick_velocity_maxwell(v: float) -> float:
    """
    Maxwell probability density function for the neutron stars' initial kick
    velocity magnitude following Hobbs et al. (2005).

    Args:
        v (float): initial kick velocity magnitude in [km/s].

    Returns:
        float: stellar kick velocity distribution in [1/(km/s)].
    """
    sigma = 265.
    pdf_vk = (
        np.sqrt(2 / np.pi)
        * v ** 2
        / (sigma ** 3)
        * np.exp(-(v ** 2) / (2 * sigma ** 2))
    )

    return pdf_vk

In [ ]:
v_tot = np.sqrt(v_r**2 + v_phi**2 + v_z**2)

fig, ax = plt.subplots()
v_bins = np.linspace(0,1500.,51)  

ax.hist(
    v_tot,
    bins=v_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=0.7,
    label=r"Simulated",
    density=True
)
ax.plot(
    v_bins,
    pdf_kick_velocity_maxwell(v_bins),
    linestyle="-",
    lw=4,
    color="tab:red",
    alpha=0.7,
    label=r"Maxwell $\sigma = 265$ km s$^{-1}$",
)
ax.set_xlabel(r"Total velocity magnitude [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.legend(frameon=False, loc=0)

plt.show()

Plotting the distribution of simulated pulsars in RA and DEC in the ICRS (International Celestial Reference System) frame and comparing with neutron stars in the ATNF catalogue. We select from the ATNF catalog all neutron stars that are presumably isolated and not recycled. We also compare with the sub-sample of those that have a measured proper motion.

In [ ]:
# Read full atnf catalog (binaries excluded) .csv file.
data_atnf = pd.read_csv("../data/atnf_full_nobinary_13-11-2020.csv", delimiter=',', header=[0,1])
data_atnf.head()

In [ ]:
# Select only stars with measure of P and Pdot and that are not in globular clusters or in the Magellanic Clouds
data_atnf = data_atnf[~data_atnf["P0"]["[s]"].isin(['NAN'])]
data_atnf = data_atnf[~data_atnf["P1"]["[s/s]"].isin(['NAN'])]
#data_atnf = data_atnf[~data_atnf["DIST"]["(kpc)"].isin(['NAN'])]
data_atnf = data_atnf[~data_atnf["ASSOC"]["Unnamed: 21_level_1"].isin(['EXGAL:SMC', 'EXGAL:LMC', 'GC:47Tuc', 'GC:M3', 'GC:M5', 'GC:M13', 'GC:NGC6440', 'GC:Ter5', 'GC:NGC6441', 'GC:NGC6517', 'GC:NGC6522', 'GC:NGC6624', 'GC:M28(NGC6626)', 'GC:NGC6652', 'GC:M22(NGC6656)', 'GC:NGC6752', 'GC:NGC6760', 'GC:M15', 'GC:M30'])]

RA_atnf = data_atnf["RAJD"]["[deg]"].to_numpy().astype(np.float) 
DEC_atnf = data_atnf["DECJD"]["[deg]"].to_numpy().astype(np.float) 
P_atnf = data_atnf["P0"]["[s]"].to_numpy().astype(np.float) 
Pdot_atnf = data_atnf["P1"]["[s/s]"].to_numpy().astype(np.float) 
#dist_atnf = data_atnf["DIST"]["[kpc]"].to_numpy().astype(np.float)

# select only isolated non recycled neutron stars i.e. with Pdot > 1e-17
cond = (Pdot_atnf > 1e-17)
RA_atnf = RA_atnf[cond]
DEC_atnf = DEC_atnf[cond]

In [ ]:
# Read observed proper motion neutron stars .csv file
data_pm = pd.read_csv("../data/PSRs_prop_motion_22-05-2020.csv", header=[0,1])
data_pm.head()

# Select only stars with measure of P and Pdot and that are not in globular clusters
data_pm = data_pm[~data_pm["P0"]["[s]"].isin(['NAN'])]
data_pm = data_pm[~data_pm["P1"]["[s/s]"].isin(['NAN'])]
#data_pm = data_pm[~data_pm["DIST_DM"]["[kpc]"].isin(['NAN'])]
data_pm = data_pm[~data_pm["ASSOC"]["Unnamed: 24_level_1"].isin(['EXGAL:SMC', 'EXGAL:LMC', 'GC:47Tuc', 'GC:M3', 'GC:M5', 'GC:M13', 'GC:NGC6440', 'GC:Ter5', 'GC:NGC6441', 'GC:NGC6517', 'GC:NGC6522', 'GC:NGC6624', 'GC:M28(NGC6626)', 'GC:NGC6652', 'GC:M22(NGC6656)', 'GC:NGC6752', 'GC:NGC6760', 'GC:M15', 'GC:M30'])]

# Extract parameters
RA_pm = data_pm["RAJD"]["[deg]"].to_numpy().astype(np.float) 
DEC_pm = data_pm["DECJD"]["[deg]"].to_numpy().astype(np.float) 
pmRA_pm = data_pm["PMRA"]["[mas/yr]"].to_numpy().astype(np.float) 
pmRA_err_pm = data_pm["PMRA_err"]["[mas/yr]"].to_numpy().astype(np.float) 
pmDEC_pm = data_pm["PMDEC"]["[mas/yr]"].to_numpy().astype(np.float) 
pmDEC_err_pm = data_pm["PMDEC_err"]["[mas/yr]"].to_numpy().astype(np.float) 
#dist_pm = data_pm["DIST_DM"]["[kpc]"].to_numpy().astype(np.float) 
NS_class = data_pm["CLASS"]["Unnamed: 13_level_1"].to_numpy()
P_pm = data_pm["P0"]["[s]"].to_numpy().astype(np.float) 
Pdot_pm = data_pm["P1"]["[s/s]"].to_numpy().astype(np.float) 
assoc = data_pm["ASSOC"]["Unnamed: 24_level_1"].to_numpy()

# select only isolated non recycled neutron stars i.e. with Pdot > 1e-17
cond = (Pdot_pm > 1e-17) & (NS_class != "Binary PSR")
RA_pm = RA_pm[cond]
DEC_pm = DEC_pm[cond]
pmRA_pm = pmRA_pm[cond]
pmRA_err_pm = pmRA_err_pm[cond]
pmDEC_pm = pmDEC_pm[cond]
pmDEC_err_pm = pmDEC_err_pm[cond]
#dist_pm = dist_pm[cond]

In [ ]:
RA_galcen = 266.4
DEC_galcen = -29.0

fig, ax = plt.subplots(figsize=(15,8))

ax.plot(
    RA, 
    DEC, 
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=3,
    alpha=0.3,
    rasterized=True,
    label=r"Simulated",
)
ax.plot(
    RA_atnf, 
    DEC_atnf, 
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Observed full ATNF",
)
ax.plot(
    RA_pm, 
    DEC_pm, 
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Observed proper motion",
)


ax.plot(RA_galcen, DEC_galcen, marker="*", color="tab:red", markersize=20)
ax.set_xlim(0., 360.)
ax.set_ylim(-90., 90.)
ax.set_xlabel('RA [deg]')
ax.set_ylabel('DEC [deg]')
ax.legend(frameon=True, loc=0)

plt.show()

Histrogramming the simulated pulsars RA and DEC coordinate position and comparing with the observed neutron stars in the ATNF catalogue.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

RA_bins = np.linspace(0., 360., 50)

ax.hist(
    RA,
    bins=RA_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated",
)
ax.hist(
    RA_atnf,
    bins=RA_bins,
    histtype="stepfilled",
    edgecolor="tab:green",
    facecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Observed full ATNF",
)
ax.hist(
    RA_pm,
    bins=RA_bins,
    histtype="stepfilled",
    edgecolor="tab:orange",
    facecolor="tab:orange",
    lw=4,
    alpha=1,
    label=r"Observed proper motion",
)
ax.set_xlabel(r"RA [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale('log')
ax.set_xlim(0., 360.)
ax.legend(frameon=False, loc=0)

plt.show()

In [ ]:
fig, ax = plt.subplots()

DEC_bins = np.linspace(-90., 90., 25)

ax.hist(
    DEC,
    bins=DEC_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated",
)
ax.hist(
    DEC_atnf,
    bins=DEC_bins,
    histtype="stepfilled",
    edgecolor="tab:green",
    facecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Observed full ATNF",
)
ax.hist(
    DEC_pm,
    bins=DEC_bins,
    histtype="stepfilled",
    edgecolor="tab:orange",
    facecolor="tab:orange",
    lw=4,
    alpha=1,
    label=r"Observed proper motion",
)
ax.set_xlabel(r"DEC [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale('log')
ax.set_xlim(-90., 90.)
ax.legend(frameon=False, loc=0)

plt.show()

Histrogramming the simulated pulsars angular proper velocity in RA and DEC and comparing with the proper velocity of the observed neutron stars.

In [ ]:
fig, ax = plt.subplots()

x_bins = np.linspace(-200, 200, 50)

ax.hist(
    v_RA,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated",
)
ax.hist(
    pmRA_pm,
    bins=x_bins,
    histtype="stepfilled",
    edgecolor="tab:orange",
    facecolor="tab:orange",
    lw=4,
    alpha=1,
    label=r"Observed",
)
ax.set_xlabel(r"$\mu_{\rm RA}$ [mas yr$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale('log')
ax.legend(frameon=False, loc=0)

plt.show()

In [ ]:
fig, ax = plt.subplots()

x_bins = np.linspace(-200, 200, 50)

ax.hist(
    v_DEC,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated",
)
ax.hist(
    pmDEC_pm,
    bins=x_bins,
    histtype="stepfilled",
    edgecolor="tab:orange",
    facecolor="tab:orange",
    lw=4,
    alpha=1,
    label=r"Observed",
)
ax.set_xlabel(r"$\mu_{\rm DEC}$ [mas yr$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
ax.legend(frameon=False, loc=0)
ax.set_yscale('log')

plt.show()